# AsthmaAware on Kaggle

Enable **Internet** and a **GPU** in Kaggle notebook settings. Push your desired
code to GitHub before running; this notebook clones `main` by default.
Private repositories require authenticated Git access.

The notebook calls the existing `asthma_aware.prepare_data` and
`asthma_aware.train_model` commands. Preparation, normalization, model creation,
training, validation, and checkpoint saving stay in those Python modules.

Default training sources are temperature, dewpoint, and NO2. O3 is currently
blocked by a missing `totalcol_o3` GEOS baseline. The inputs are GEOS analyses;
results measure analysis downscaling rather than operational forecast skill.
Set `EPOCHS = 1` initially to check the complete workflow.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os
import sys
import subprocess

REPO_URL = "https://github.com/gt-big-data/asthma-aware.git"
BRANCH = "main"
SOURCES = ["temperature", "dewpoint", "no2"]
EPOCHS = 20
LEARNING_RATE = 1e-3

LOCAL_ROOT = Path("/kaggle/working/asthma-aware")
ML_ROOT = LOCAL_ROOT / "ml-work"
os.environ["HF_HOME"] = "/tmp/asthma-aware-hf"

if not LOCAL_ROOT.exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH,
         REPO_URL, str(LOCAL_ROOT)],
        check=True,
    )
else:
    assert (LOCAL_ROOT / ".git").is_dir(), f"Not a Git checkout: {LOCAL_ROOT}"
    print("Using the existing checkout; it is not automatically updated.")

subprocess.run(["git", "log", "-1", "--oneline"], cwd=LOCAL_ROOT, check=True)
RUN_DIR = (
    Path("/kaggle/working/training_runs")
    / datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_%f")
)
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Run output:", RUN_DIR)

## Install the project

Use Kaggle's GPU-enabled PyTorch installation when it already satisfies the
project requirements. The existing trainer uses one GPU and batch size 1.

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", f"{ML_ROOT}[data]"],
    check=True,
)

import torch

assert torch.cuda.is_available(), "Enable a GPU in Kaggle notebook settings."
print("GPU:", torch.cuda.get_device_name(0))

## Run the existing preparation command

Rebuild the catalog and training-only normalization on Kaggle because prepared
indexes contain machine-specific paths and fingerprints. This runs the original
configuration for all configured sources. Missing satellite rasters and masks use
the existing Hugging Face fallback. Blocked sources are reported by the command;
they cannot be included in `SOURCES` for training.

In [ ]:
subprocess.run(
    [sys.executable, "-u", "-m", "asthma_aware.prepare_data",
     "--config", str(ML_ROOT / "data_config.json"),
     "--output", str(RUN_DIR / "prepared"),
     "--fit-normalization"],
    cwd=ML_ROOT,
    check=True,
)

## Run the existing training command

The trainer reads `prepared/` and saves best validation checkpoints in
`checkpoints/`, relative to its working directory. Running it inside `RUN_DIR`
places both under Kaggle's output directory without changing its path handling.
The trainer manages validation and saves each source's best model. Rerunning
this cell starts fresh training; it does not automatically resume checkpoints.

In [ ]:
subprocess.run(
    [sys.executable, "-u", "-m", "asthma_aware.train_model",
     "--epochs", str(EPOCHS),
     "--sources", *SOURCES,
     "--learning-rate", str(LEARNING_RATE)],
    cwd=RUN_DIR,
    check=True,
)

## Download outputs

The archive contains the best model checkpoints and prepared data metadata.
Prepared files contain Kaggle-specific paths; rebuild them on another machine.
Download the archive or save a Kaggle notebook version with outputs included.

Kaggle documentation: https://www.kaggle.com/docs/notebooks

In [ ]:
import shutil
from IPython.display import FileLink, display

archive = shutil.make_archive(str(RUN_DIR), "zip", root_dir=RUN_DIR)
print("Checkpoints:", RUN_DIR / "checkpoints")
print("Download:", archive)
os.chdir("/kaggle/working")
display(FileLink(str(Path(archive).resolve().relative_to(Path.cwd()))))